![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform Embeddings & Vector Search on Redis

In this recipe we register **embedding features** in [**Featureform**](https://docs.featureform.com/), materialize them to **Redis as a vector index**, and run **semantic (nearest-neighbor) search** against them with `client.nearest()`.

## Why embeddings belong in a feature store
An embedding is just a feature whose value is a vector. Treating it as a first-class Featureform feature buys you the same guarantees as any other feature: it's **defined once**, **versioned**, and **served from a low-latency online store** — here, Redis, which doubles as the vector index for similarity search. The model that produced the embedding, the source rows, and the serving index all stay linked.

## What we'll build
A tiny **semantic product search**:
1. Embed product descriptions with a sentence-transformer model.
2. Load the vectors into ClickHouse and register them as an `ff.Embedding` feature, materialized to **Redis**.
3. Embed a free-text query and ask Redis for the nearest products with `client.nearest()`.

> ℹ️ **Why the embeddings are precomputed in Python:** Featureform runs SQL transformations in the offline store, and SQL can't call a transformer model. So we compute vectors in the notebook and load them into ClickHouse. (Computing embeddings *inside* a transformation would require a Spark/Kubernetes provider.)

## The stack — all local, no Spark

- **ClickHouse** — offline store; holds the source rows and their precomputed vectors.
- **Redis** — online store **and vector index**; serves nearest-neighbor queries.
- **Featureform** coordinator — registers resources and materializes the vectors into Redis.

> ⚠️ **Needs local Docker; will not run on Colab or in CI.** The next cell starts all three containers itself (coordinator on gRPC `localhost:7878`, dashboard `http://localhost`) and the cleanup cell removes them — you don't need anything running beforehand.

### Start the local stack (Featureform coordinator, ClickHouse, Redis)

In [1]:
# NBVAL_SKIP
# Launch the full local stack this notebook needs on a private Docker network, so
# the coordinator reaches ClickHouse/Redis by container name and we don't have to
# publish (and risk host-port collisions on) their internal ports. Only the ports
# the host itself uses are published: ClickHouse HTTP 8123 (data load) and the
# coordinator's gRPC 7878 + dashboard 80. Remove any existing containers/network
# first so a re-run always gets a fresh registry and store. (Other systems may
# start/stop these containers; we own their lifecycle here.)
!docker rm -f featureform clickhouse redis 2>/dev/null
!docker network rm ff-net 2>/dev/null
!docker network create ff-net
!docker run -d --name clickhouse --network ff-net -p 8123:8123 -e CLICKHOUSE_PASSWORD=featureform clickhouse/clickhouse-server:latest
!docker run -d --name redis --network ff-net redis:8
!docker run -d --name featureform --network ff-net -p 80:80 -p 7878:7878 featureformcom/featureform:latest

# Wait until everything is ready. ClickHouse needs a few seconds to apply the
# password (it rejects auth during that window) and the coordinator validates the
# ClickHouse provider on apply(), so both must be reachable before we register.
import subprocess, time

def _ready():
    ch = subprocess.run(["docker", "exec", "clickhouse", "clickhouse-client",
                         "--password", "featureform", "-q", "SELECT 1"],
                        capture_output=True, text=True)
    rd = subprocess.run(["docker", "exec", "redis", "redis-cli", "ping"],
                        capture_output=True, text=True)
    ff = subprocess.run(["docker", "inspect", "--format",
                         "{{.State.Health.Status}}", "featureform"],
                        capture_output=True, text=True)
    return (ch.stdout.strip() == "1" and rd.stdout.strip() == "PONG"
            and ff.stdout.strip() == "healthy")

for _ in range(150):
    if _ready():
        print("featureform + clickhouse + redis ready")
        break
    time.sleep(2)
else:
    raise RuntimeError("stack did not become ready in time; check `docker ps -a`")

bd36809e85fc2cb808e82395e7eb3de65b7871b8c0a69954c282caae399d71ca
0fb6c973fd01870ada20c05c01e0fc6717f6290ffc217d3b09787d2e1b084063
d5aea8ee352ea46cd9e369c307322bfa1356929962d0c9c85f68907be72f6755
9b0c5a27d92ad988d452e6be488b9f20aaedcc9b77bf73b86ad8dd928cded741
featureform + clickhouse + redis ready


## Environment Setup

### Install Python Dependencies

In [2]:
%pip install -q featureform redis clickhouse-connect sentence-transformers pandas

Note: you may need to restart the kernel to use updated packages.


### Configure connections

The coordinator container reaches ClickHouse and Redis by **container name** over the shared `ff-net` network. This notebook (running on the host) talks to ClickHouse's published HTTP port and the coordinator's published gRPC port via `localhost`.

In [3]:
import os

# Featureform coordinator (gRPC), reached from this notebook on the host.
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

# The coordinator reaches the providers by container name over the shared Docker
# network (ff-net), so these are container names + internal ports, not host ports.
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "clickhouse")
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "featureform")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

REDIS_HOST = os.getenv("REDIS_HOST", "redis")
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

### Embed products and load them into ClickHouse

We embed each product description with `all-MiniLM-L6-v2` (384-dimensional vectors) and store the vectors in a ClickHouse `Array(Float32)` column. `DIMS` must match both the model and the `ff.Embedding` definition later.

In [4]:
# NBVAL_SKIP
import clickhouse_connect
from sentence_transformers import SentenceTransformer

PRODUCTS = [
    ("p01", "Wireless noise-cancelling over-ear headphones"),
    ("p02", "Bluetooth portable speaker, waterproof"),
    ("p03", "Ergonomic mechanical keyboard with RGB backlight"),
    ("p04", "4K ultra-wide gaming monitor, 144Hz"),
    ("p05", "Stainless steel insulated water bottle"),
    ("p06", "Cast iron skillet, pre-seasoned"),
    ("p07", "Trail running shoes with grip sole"),
    ("p08", "Merino wool hiking socks, 3-pack"),
]

model = SentenceTransformer("all-MiniLM-L6-v2")
DIMS = model.get_sentence_embedding_dimension()  # 384

ids = [p[0] for p in PRODUCTS]
names = [p[1] for p in PRODUCTS]
vectors = model.encode(names).tolist()

# ClickHouse HTTP port 8123 is published to the host, so we load from localhost.
ch = clickhouse_connect.get_client(host="localhost", port=8123,
                                   username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD)
ch.command("DROP TABLE IF EXISTS products")
ch.command(
    """
    CREATE TABLE products (
        id String,
        name String,
        embedding Array(Float32)
    ) ENGINE = MergeTree ORDER BY id
    """
)
ch.insert("products", list(zip(ids, names, vectors)),
          column_names=["id", "name", "embedding"])
print(f"loaded {len(ids)} products, {DIMS}-dim embeddings")

/Users/justin.cechmanek/.pyenv/versions/redis-ai-res/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded 8 products, 384-dim embeddings


## Register the providers

In [5]:
import featureform as ff

# Workaround for bugs in featureform 1.15.8 (latest release) that break the
# embedding/vector-search path only:
#   1. featureform/types.py uses `pb` but never imports it -> NameError on apply()
#   2. VectorType.from_proto references an undefined `protoVal` -> NameError
#   3. client.nearest() calls impl._nearest, but the method is named `nearest`
import featureform.types as _ff_types
from featureform.proto import metadata_pb2 as _ff_pb
from featureform.enums import ScalarType as _ff_ScalarType
from featureform.serving import HostedClientImpl as _ff_hosted
_ff_types.pb = _ff_pb
_ff_hosted._nearest = _ff_hosted.nearest
_ff_types.VectorType.from_proto = classmethod(
    lambda cls, v: cls(_ff_ScalarType.from_proto(v.scalar), v.dimension, v.is_embedding)
)

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store holding product vectors",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

## Register the source and the embedding feature

We register the `products` table, then declare an `ff.Embedding` over its vector column. `vector_db=redis` tells Featureform to materialize the vectors into Redis and build a vector index there; `dims` must match the model. `@ff.entity` keys the embedding by product.

In [6]:
products = clickhouse.register_table(
    name="products",
    variant="quickstart",
    table="products",
)

@ff.entity
class Product:
    product_embedding = ff.Embedding(
        products[["id", "embedding"]],
        dims=DIMS,
        vector_db=redis,
        variant="quickstart",
        description="Sentence-transformer embedding of the product description",
    )

## Apply

`client.apply()` registers everything and materializes the vectors into Redis, building the searchable index.

In [7]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

UserWarning: install "ipywidgets" for Jupyter support

Applying Run: 2026-07-24t15-46-15
Creating User default_owner 
Creating Provider clickhouse-quickstart 
Creating Provider redis-quickstart 
Creating Source Variant products quickstart
Creating Entity product 
Creating Feature Variant product_embedding quickstart



## Semantic search from Redis

Embed a free-text query with the **same model**, then ask Redis for the nearest product embeddings. `client.nearest()` returns the entity keys (product ids) of the closest vectors — served from the Redis index.

In [8]:
# NBVAL_SKIP
import time

query = "something to listen to music outdoors"
query_vec = model.encode(query).tolist()

# Materialization into Redis is eventually consistent: apply() can return just
# before the vectors are queryable. Retry briefly until the index answers.
for attempt in range(10):
    try:
        neighbors = client.nearest(("product_embedding", "quickstart"), query_vec, k=3)
        break
    except Exception:
        if attempt == 9:
            raise
        time.sleep(2)

name_by_id = dict(zip(ids, names))
print(f"query: {query!r}\n")
for pid in neighbors:
    print(f"  {pid}: {name_by_id.get(pid, '?')}")

query: 'something to listen to music outdoors'

  p02: Bluetooth portable speaker, waterproof
  p01: Wireless noise-cancelling over-ear headphones
  p03: Ergonomic mechanical keyboard with RGB backlight


### What just happened

The nearest neighbors came back from **Redis**, not from re-scanning the source. The embedding is a normal Featureform feature — versioned and defined once — that happens to be served through a vector index. The dashboard at **http://localhost** shows it alongside every other feature, with its source lineage intact.

## Cleanup

Remove the coordinator and both stores. Because the setup cell tears these down and recreates them, you can re-run this notebook top-to-bottom any time and get a clean, fully materialized run.

In [9]:
# NBVAL_SKIP
# Remove everything this notebook started (coordinator, both stores, network).
!docker rm -f featureform clickhouse redis
!docker network rm ff-net

featureform
clickhouse
redis
ff-net


## Learn more

- [Featureform embeddings & vector search](https://docs.featureform.com/)
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb)
- [RedisVL vector search recipes](../vector-search/) — using Redis as a vector database directly